# 03. Model Training, Deep Sequence Learning & Comparative Evaluation
**Models Trained:**
1. Linear Ridge Baseline
2. Random Forest Regressor
3. Tuned XGBoost
4. PyTorch 2-Layer Bidirectional LSTM (Bi-LSTM)

**Warning Horizon:** 24-Hour Forward Lead Time


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.models.train_baselines import run_baseline_experiments
from src.models.train_lstm import train_bilstm
from pathlib import Path

print("Evaluation environment initialized!")


## 1. Training & Benchmarking Tabular Models (Ridge, RF, XGBoost)

In [ ]:
data_path = Path("../data/processed/monsoon_features_2022.csv")
output_dir = Path("../outputs")

baseline_results = run_baseline_experiments(data_path, output_dir)
baseline_results


## 2. Training Deep Sequence Forecaster: Bidirectional LSTM (Bi-LSTM)
Windowing: 7-day lookback sequence with dropout to prevent overfitting on convective extremes.


In [ ]:
bilstm_res, actuals, preds = train_bilstm(
    data_path=data_path,
    output_dir=output_dir,
    seq_len=7,
    epochs=80
)


## 3. Publication Results Summary (Lead Time: 24h)

In [ ]:
all_results = baseline_results.copy()
all_results.loc[len(all_results)] = bilstm_res
all_results = all_results.sort_values(by="RMSE (mm)").reset_index(drop=True)
print("=== FINAL COMPARATIVE PERFORMANCE TABLE ===")
display(all_results)


In [ ]:
# Plotting Prediction Curves vs Actuals
plt.figure(figsize=(11, 4.5))
plt.plot(actuals, label='Ground Truth 24h Rainfall (mm)', color='black', lw=2)
plt.plot(preds, label='Bi-LSTM Forecasted Rainfall (mm)', color='#1f77b4', lw=2, linestyle='--')
plt.title("Held-Out Test Sequence: Bi-LSTM 24-Hour Lead Forecast vs Actuals")
plt.xlabel("Test Sample Days")
plt.ylabel("Rainfall (mm/day)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
